# D1 · Reading the decision boundary: roughness, regions and k

**Type**: Interpret · **Level**: ●●○ independent · **Time**: about 45 minutes · **core path ★**

## Objective

Turn the KNN decision boundary from an image into **numbers**: for nine values of `k` compute on a grid the roughness of the boundary (fraction of adjacent cells with different classes) and the number of connected regions, and read from the data
where there is overfitting, where the model is too rigid and how large `k` must be to "simplify" the boundary.

## Dataset

`make_moons(600, noise=0.3)` split 50/50 (300 training examples, 300 test ones). `grid_map(k)` (given) returns the prediction of the model on a 100×100 grid covering the plane (matrix of 0s and 1s).

## Educational purpose

Learn to read a decision boundary without trusting the eye: the number of connected regions and the length of the boundary are measures of the complexity of the model, and `k` is its only regulator.

## Task

1. For `k` in `[1, 3, 5, 9, 15, 31, 61, 121, 201]` build `table` (index `k`, columns `train_acc`, `test_acc`, `roughness`, `n_regions`). `roughness`: fraction of pairs of adjacent cells (horizontal and vertical) of the map with different classes. `n_regions`: number of connected components of the map (count the regions of 0 and of 1 with `scipy.ndimage.label`, 4-connectivity).
2. Save `overfit_k` (the `k` with the largest difference `train_acc − test_acc`) and `best_k` (the `k` with maximum `test_acc`; on a tie the smallest).
3. Save `k1_regions` (number of regions with `k = 1`) and `two_region_k` (the smallest `k` whose map has exactly **two** regions).
4. Answer with booleans: `roughness_decreases` (the roughness **never** increases as `k` grows: true or false?), `train_below_test_at_large_k` (is there a `k` where `train_acc < test_acc`?), `k1_has_train_one` (the training accuracy of `k = 1` is 1).

## Setup

Run this cell without changing it.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

from scipy.ndimage import label
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier

X, y = make_moons(600, noise=0.3, random_state=0)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.5, random_state=0)
g1, g2 = np.meshgrid(np.linspace(-2.5, 3.5, 100), np.linspace(-1.8, 2.3, 100))
points = np.c_[g1.ravel(), g2.ravel()]
ks = [1, 3, 5, 9, 15, 31, 61, 121, 201]


def grid_map(k):
    """KNN prediction (k neighbors) on a 100 x 100 grid: matrix of 0s and 1s."""
    return KNeighborsClassifier(k).fit(X_tr, y_tr).predict(points).reshape(100, 100)

## Your work

In [ ]:
rows = {}
for k in ks:
    G = grid_map(k)
    rows[k] = ...
table = pd.DataFrame(rows).T
table.round(3)

In [ ]:
overfit_k = None
best_k = None
k1_regions = None
two_region_k = None

In [ ]:
roughness_decreases = None
train_below_test_at_large_k = None
k1_has_train_one = None

## Check

Run the cell when you think you are done. If a check fails, the message tells you what is wrong. If it fails a second time, a hint appears under the message.

In [ ]:
with verify(hint='Roughness: `((G[:, 1:] != G[:, :-1]).sum() + (G[1:, :] != G[:-1, :]).sum()) / (G[:, 1:].size + G[1:, :].size)`. Regions: `label(G == 0)[1] + label(G == 1)[1]`. Each row: `{"train_acc": m.score(X_tr, y_tr), ...}` with `m = KNeighborsClassifier(k).fit(X_tr, y_tr)`.'):
    check(list(table.columns) == ["train_acc", "test_acc", "roughness", "n_regions"] and list(table.index) == ks, "table with the four columns for nine values of k", "index k, columns train_acc, test_acc, roughness, n_regions")
    G1 = grid_map(1)
    rough_ref = ((G1[:, 1:] != G1[:, :-1]).sum() + (G1[1:, :] != G1[:-1, :]).sum()) / (G1[:, 1:].size + G1[1:, :].size)
    check(close(table.loc[1, "roughness"], rough_ref), "roughness computed as the fraction of different adjacent pairs", "compare G[:, 1:] with G[:, :-1] and G[1:, :] with G[:-1, :]")
    check(table.loc[1, "n_regions"] == label(G1 == 0)[1] + label(G1 == 1)[1], "number of regions: connected components of 0 and of 1", "label(G == 0)[1] + label(G == 1)[1]")
    check(overfit_k == (table["train_acc"] - table["test_acc"]).idxmax() and overfit_k == 1, "the maximum train-test gap is at k = 1", "idxmax of the difference")
    check(best_k == table["test_acc"].idxmax() and 9 <= best_k <= 61, "the best k on the test set is intermediate (between 9 and 61)", "idxmax of test_acc")
    check(k1_regions >= 8 and two_region_k <= 15 and table.loc[201, "n_regions"] == 2, "with k = 1 at least 8 regions; from k = 15 at most two; with k = 201 exactly two", "n_regions in table")
    check(roughness_decreases is False and table.loc[1, "roughness"] > 1.5 * table.loc[15, "roughness"] and abs(table.loc[15, "roughness"] - table.loc[61, "roughness"]) < 0.002, "the roughness does NOT always fall: it drops until k = 15 (at k = 1 it is over 1.5 times) and then stays on a plateau", "look at the roughness column: is_monotonic_decreasing")
    check(k1_has_train_one is True and train_below_test_at_large_k is False and abs(table.loc[201, "train_acc"] - table.loc[201, "test_acc"]) < 0.02 and table.loc[201, "test_acc"] < table["test_acc"].max() - 0.08, "k = 1: perfect training; the training is never below the test; at k = 201 train and test coincide but are both low (underfitting)", "compare train_acc and test_acc row by row")
done()

---
The commented solution is in `solutions/D_interpret/D1_decision_boundary_and_k.ipynb`. Open it after you pass the check, or after you have really tried.